In [29]:
pip install mlflow

In [ ]:
# In order to use the MLflowClient API, the initial step involves importing the necessary modules
import pandas as pd # <-- ¡Asegúrate de que esta esté importada!
import numpy as np  # <-- ¡Asegúrate de que esta esté importada para np.sqrt!
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [31]:
# Por default las modificaciones a  MLFLOW tracking URI se guardan en el almacenamiento local as stracking server.
#Sin embargo aqui se va a usar el tracking server iniciado en la documentacion en vez de guardar en la documentación local
# Usar 0.0.0.0.0, el otro no funciona
#URI = Uniform Resource Identifier, es decir, una forma estándar de identificar un recurso
# (como una dirección web, archivo local, o base de datos).
client = MlflowClient(tracking_uri="http://127.0.0.1:8080")


In [32]:
#Search experiments:
all_experiments = client.search_experiments()

print(all_experiments)


[<Experiment: artifact_location='mlflow-artifacts:/297784242467499909', creation_time=1762021299832, experiment_id='297784242467499909', last_update_time=1762021299832, lifecycle_stage='active', name='Apple_Models_3', tags={'mlflow.experimentKind': 'custom_model_development'}>, <Experiment: artifact_location='mlflow-artifacts:/415594987185625826', creation_time=1762013543258, experiment_id='415594987185625826', last_update_time=1762013543258, lifecycle_stage='active', name='Apple_Models', tags={'mlflow.experimentKind': 'custom_model_development',
 'mlflow.note.content': 'This is the grocery forecasting project. This '
                        'experiment contains the produce models for apples.',
 'project_name': 'grocery-forecasting',
 'project_quarter': 'Q3-2023',
 'store_dept': 'produce',
 'team': 'stores-ml'}>, <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1762006707203, experiment_id='0', last_update_time=1762006707203, lifecycle_stage='active', name='Default',

In [ ]:
#To get familiar with accessing elements from returned collections from MLflow APIs, 
# let's extract the name and the lifecycle_stage from the search_experiments() 
# query and extract these attributes into a dict.


default_experiment = [
    {"name": experiment.name, "lifecycle_stage": experiment.lifecycle_stage}
    for experiment in all_experiments
    if experiment.name == "Default"
][0]

print(default_experiment)

{'lifecycle_stage': 'active', 'name': 'Default'}


In [34]:
#Crea un experimento

# Provide an Experiment description that will appear in the UI
experiment_description = (
    "This is the grocery forecasting project. "
    "This experiment contains the produce models for apples."
)

# Provide searchable tags that define characteristics of the Runs that
# will be in this Experiment
experiment_tags = {
    "project_name": "grocery-forecasting",
    "store_dept": "produce",
    "team": "stores-ml",
    "project_quarter": "Q3-2023",
    "mlflow.note.content": experiment_description,
}

# Create the Experiment, providing a unique name
produce_apples_experiment = client.create_experiment(
    name="Apple_Models", tags=experiment_tags
)

RestException: RESOURCE_ALREADY_EXISTS: Experiment 'Apple_Models' already exists.

In [37]:
# Use search_experiments() to search on the project_name tag key
#Se busca usando el nombre del proyecto

apples_experiment = client.search_experiments(
    filter_string="tags.`project_name` = 'grocery-forecasting'"
)

print(vars(apples_experiment[0]))

{'_experiment_id': '415594987185625826', '_name': 'Apple_Models', '_artifact_location': 'mlflow-artifacts:/415594987185625826', '_lifecycle_stage': 'active', '_tags': {'mlflow.experimentKind': 'custom_model_development', 'mlflow.note.content': 'This is the grocery forecasting project. This experiment contains the produce models for apples.', 'project_name': 'grocery-forecasting', 'project_quarter': 'Q3-2023', 'store_dept': 'produce', 'team': 'stores-ml'}, '_creation_time': 1762013543258, '_last_update_time': 1762013543258}


In [38]:
print(apples_experiment[0].tags["team"])

stores-ml


In [47]:
import pandas as pd # <-- ¡Asegúrate de que esta esté importada!
import numpy as np  # <-- ¡Asegúrate de que esta esté importada para np.sqrt!
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Sets the fluent API to set the tracking uri and the active experiment
mlflow.set_tracking_uri("http://127.0.0.1:8080")

# Sets the current active experiment to the 'Apple_Models' experiment and returns the Experiment metadata
apple_experiment = mlflow.set_experiment('Apple_Models_3')

# Define a run name for this iteration of training.
# If this is not set, a unique name will be auto-generated for your run.
run_name = 'apples_rf_test_3_1'



In [48]:
with mlflow.start_run(run_name=run_name) as run:
    
    # 1. Lectura de Datos
    direccion="C:\\Users\\Anuar\\Documents\\Maestria Inteligencia artificial\\MLOPs Prueba\\Cookie cutter\\proyecto_mlops_equipo_56_fase_2\\data\\external\\dataset_apples.csv"
    data = pd.read_csv(direccion) # Ya no necesitas df_ventas, usa 'data' directamente
    
    # 2. Split
    X = data.drop(columns=['date', 'demand'])
    y = data['demand']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    params = {
        'n_estimators': 100,
        'max_depth': 6,
        'min_samples_split': 10,
        'min_samples_leaf': 4,
        'bootstrap': True,
        'oob_score': False,
        'random_state': 888
    }

    # 3. Entrenamiento
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)

    # 4. Evaluación
    y_pred = rf.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse) # <-- Requiere 'import numpy as np'
    r2 = r2_score(y_val, y_pred)

    metrics = {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

    # 5. REGISTRO
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    
    # --- LA CLAVE ---
    # Usamos X_train.iloc[0:2] para asegurar que el input_example sea un pequeño DataFrame válido.
    mlflow.sklearn.log_model(
        sk_model=rf,
        input_example=X_train.iloc[0:2], # <-- Usa las primeras filas del conjunto de entrenamiento
        artifact_path='rf_apples'
    )
    print("¡Registro completado! Verifica los Artifacts en la interfaz.")

2025/11/01 14:46:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\Anuar\anaconda3\envs\env_pyspark\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


¡Registro completado! Verifica los Artifacts en la interfaz.
🏃 View run apples_rf_test_3_1 at: http://127.0.0.1:8080/#/experiments/297784242467499909/runs/b818637b580746fc9231098296c1081a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/297784242467499909
